In [1]:
import pandas as pd
import numpy as np

from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import FeatureUnion

In [2]:
df = pd.read_csv('../After_EDA_data/Light_text.csv')

In [3]:
df

,uid,profile,anime_uid,score,scores,text
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...
...,...,...,...,...,...,...
192107,240067,Unicorn819,1281,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ok this anime is pretty old but here s the bac...
192108,285777,ShizzoSVH,1281,9,"{'Overall': '9', 'Story': '7', 'Animation': '9...",the dub for this anime is made this anime a fu...
192109,286904,AlluMan96,1281,3,"{'Overall': '3', 'Story': '3', 'Animation': '1...",some might argue that doing a review of a show...
192110,287903,AgentK300,1281,10,"{'Overall': '10', 'Story': '3', 'Animation': '...",absolutely hilarious i accidentally came acros...


In [4]:
df.shape

(192112, 6)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192112 entries, 0 to 192111
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   uid        192112 non-null  int64 
 1   profile    192112 non-null  object
 2   anime_uid  192112 non-null  int64 
 3   score      192112 non-null  int64 
 4   scores     192112 non-null  object
 5   text       192112 non-null  object
dtypes: int64(3), object(3)
memory usage: 8.8+ MB


In [6]:
df['text'].duplicated().sum()

np.int64(61764)

In [7]:
df['text'] = df['text'].drop_duplicates()

In [8]:
df = df.dropna()

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 130348 entries, 0 to 182637
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   uid        130348 non-null  int64 
 1   profile    130348 non-null  object
 2   anime_uid  130348 non-null  int64 
 3   score      130348 non-null  int64 
 4   scores     130348 non-null  object
 5   text       130348 non-null  object
dtypes: int64(3), object(3)
memory usage: 7.0+ MB


In [10]:
df.shape

(130348, 6)

In [11]:
def classification(text):
    if text < 6:
        return 'Bad'
    elif text < 8:
        return 'Neutral'
    else:
        return 'Good'

In [12]:
df = df.copy()

In [13]:
df['target'] = df['score'].apply(classification)

In [14]:
df

,uid,profile,anime_uid,score,scores,text,target
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...,Good
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...,Good
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...,Neutral
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...,Good
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...,Good
...,...,...,...,...,...,...,...
182629,146535,iHitokage,2593,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",you know nothing jon snow this is how i felt a...,Good
182631,13126,Onegai,2593,7,"{'Overall': '7', 'Story': '6', 'Animation': '9...",so i finally decided to watch the kara no kyou...,Neutral
182633,127899,Murasa22,2593,10,"{'Overall': '10', 'Story': '9', 'Animation': '...",this review is based on all the movies of kara...,Good
182636,286852,srry4apologizng,2593,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ufotable s beautiful use of digital lighting e...,Good


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 130348 entries, 0 to 182637
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   uid        130348 non-null  int64 
 1   profile    130348 non-null  object
 2   anime_uid  130348 non-null  int64 
 3   score      130348 non-null  int64 
 4   scores     130348 non-null  object
 5   text       130348 non-null  object
 6   target     130348 non-null  object
dtypes: int64(3), object(4)
memory usage: 8.0+ MB


In [16]:
df['target'].value_counts()

target
Good       72670
Neutral    31474
Bad        26204
Name: count, dtype: int64

In [17]:
df_majority = df[df['target'] == 'Good']
df_minority = df[df['target'] == 'Bad']
df_neutral = df[df['target'] == 'Neutral']

In [18]:
df_reduced_majority = df_majority.sample(n=len(df_minority) + 6000, random_state=42)
# df_reduced_neutral = df_neutral.sample(len(df_minority), random_state=42)

In [19]:
# df_balanced = pd.concat([df_reduced_majority, df_minority, df_reduced_neutral]).sample(frac=1, random_state=42)
df_balanced = pd.concat([df_reduced_majority, df_minority, df_neutral]).sample(frac=1, random_state=42)

In [20]:
df_balanced

,uid,profile,anime_uid,score,scores,text,target
1926,153713,Yordi,11887,10,"{'Overall': '10', 'Story': '0', 'Animation': '...",kokoro connect when i first heard of kokoro co...,Good
32794,271301,BanjoTheBear,30654,3,"{'Overall': '3', 'Story': '0', 'Animation': '0...",this review has been adapted from my blog redd...,Bad
25755,219831,overDere,1210,6,"{'Overall': '6', 'Story': '6', 'Animation': '8...",failure loneliness heartbreak suicide depressi...,Neutral
57878,164407,ShiinachiSylande,19769,7,"{'Overall': '7', 'Story': '5', 'Animation': '8...",gonna be entirely honest here if you re impati...,Neutral
68918,43301,widowmaker,6747,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",this is just a great anime that is good for ju...,Good
...,...,...,...,...,...,...,...
35306,248384,Sakurazaki,1,8,"{'Overall': '8', 'Story': '7', 'Animation': '1...",to begin i will say that i rarely watch anime ...,Good
148315,256759,LoliWitch,35248,3,"{'Overall': '3', 'Story': '3', 'Animation': '5...",i don t usually say negative stuff about the s...,Bad
101580,243170,PicaroJr,10588,6,"{'Overall': '6', 'Story': '10', 'Animation': '...",just play the fucking game don t watch this st...,Neutral
21309,80248,Animeluvr3,853,9,"{'Overall': '9', 'Story': '10', 'Animation': '...",at first i was a bit reluctant to watch this i...,Good


In [21]:
df_balanced = df_balanced.reset_index(drop=True)

In [22]:
df_balanced

,uid,profile,anime_uid,score,scores,text,target
0,153713,Yordi,11887,10,"{'Overall': '10', 'Story': '0', 'Animation': '...",kokoro connect when i first heard of kokoro co...,Good
1,271301,BanjoTheBear,30654,3,"{'Overall': '3', 'Story': '0', 'Animation': '0...",this review has been adapted from my blog redd...,Bad
2,219831,overDere,1210,6,"{'Overall': '6', 'Story': '6', 'Animation': '8...",failure loneliness heartbreak suicide depressi...,Neutral
3,164407,ShiinachiSylande,19769,7,"{'Overall': '7', 'Story': '5', 'Animation': '8...",gonna be entirely honest here if you re impati...,Neutral
4,43301,widowmaker,6747,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",this is just a great anime that is good for ju...,Good
...,...,...,...,...,...,...,...
89877,248384,Sakurazaki,1,8,"{'Overall': '8', 'Story': '7', 'Animation': '1...",to begin i will say that i rarely watch anime ...,Good
89878,256759,LoliWitch,35248,3,"{'Overall': '3', 'Story': '3', 'Animation': '5...",i don t usually say negative stuff about the s...,Bad
89879,243170,PicaroJr,10588,6,"{'Overall': '6', 'Story': '10', 'Animation': '...",just play the fucking game don t watch this st...,Neutral
89880,80248,Animeluvr3,853,9,"{'Overall': '9', 'Story': '10', 'Animation': '...",at first i was a bit reluctant to watch this i...,Good


In [23]:
df_balanced.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89882 entries, 0 to 89881
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   uid        89882 non-null  int64 
 1   profile    89882 non-null  object
 2   anime_uid  89882 non-null  int64 
 3   score      89882 non-null  int64 
 4   scores     89882 non-null  object
 5   text       89882 non-null  object
 6   target     89882 non-null  object
dtypes: int64(3), object(4)
memory usage: 4.8+ MB


In [24]:
df_balanced['target'].value_counts()

target
Good       32204
Neutral    31474
Bad        26204
Name: count, dtype: int64

In [25]:
df_balanced.duplicated().sum()

np.int64(0)

In [26]:
x = df_balanced['text']
y = df_balanced['target']

In [27]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y_encoded, random_state=42
)

In [29]:
word_tfidf = TfidfVectorizer(
    max_features = 10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=10,
    max_df=0.8
)

In [30]:
char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    max_features=10000,
    ngram_range=(3,5),
    sublinear_tf=True,
    min_df=10,
    max_df=0.8
)

In [31]:
tfidf = FeatureUnion([
    ('word', word_tfidf),
    ('char', char_tfidf)
])

In [32]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [33]:
svc = LinearSVC()

svc.fit(X_train_tfidf, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [34]:
y_pred = svc.predict(X_test_tfidf)
y_score = svc.decision_function(X_test_tfidf)

In [35]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         Bad       0.74      0.75      0.75      5241
        Good       0.76      0.80      0.78      6441
     Neutral       0.64      0.61      0.62      6295

    accuracy                           0.72     17977
   macro avg       0.72      0.72      0.72     17977
weighted avg       0.71      0.72      0.72     17977



In [36]:
print(confusion_matrix(y_test, y_pred))

[[3914  264 1063]
 [ 223 5143 1075]
 [1128 1335 3832]]
